In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

## Tools


In [8]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

#### Inbuilt Tool


In [9]:
wiki_api = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=wiki_api)
wiki.name

'wikipedia'

In [10]:
arxiv = ArxivQueryRun(
    api_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
)
arxiv.name

'arxiv'

In [11]:
tools = [wiki, arxiv]

#### Custom Tool


In [12]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
).split_documents(docs)
db = FAISS.from_documents(
    documents, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)
retriever = db.as_retriever()
retriever

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7974.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001EE95022060>, search_kwargs={})

In [14]:
from langchain_core.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever, "langsmith-search", "Search any information about Langsmith"
)

retriever_tool.name

'langsmith-search'

In [15]:
tools.append(retriever_tool)

In [16]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\ProgramData\\miniconda3\\envs\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001EE98AAF7E0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001EE98AAFC40>)]

## Agents


In [17]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="qwen/qwen3-32b")

In [18]:
from langchain_classic import hub

prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [21]:
from langchain_classic.agents import create_openai_tools_agent

agent = create_openai_tools_agent(llm, tools, prompt)

agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [22]:
from langchain_classic.agents import AgentExecutor

executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [23]:
executor.invoke({
    "input": "Tell me about Langsmith"
})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'Langsmith'}`


LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with 

{'input': 'Tell me about Langsmith',
 'output': "LangSmith is a comprehensive platform designed for building, debugging, and deploying AI agents and large language model (LLM) applications. Here's an overview of its key features and capabilities:\n\n### **Core Features**\n1. **Framework-Agnostic**: Works with any AI framework or provider, allowing flexibility in development.\n2. **Unified Workflow**: Combines observability, evaluation, prompt engineering, and deployment into a single integrated platform.\n3. **Observability**: \n   - Trace and debug applications by visualizing every step of an agent's execution.\n   - Gain insights into performance and reliability.\n4. **Evaluation**:\n   - Measure and track the quality of AI outputs over time.\n   - Ensure consistency and trustworthiness through structured evaluations.\n5. **Prompt Engineering**:\n   - Iterate on prompts with versioning and collaboration tools.\n   - Test and refine prompts to optimize results.\n\n### **Deployment & M

In [24]:
executor.invoke({
    "input": "What is Machine Learning?"
})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Machine Learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks witMachine Learning (ML) is a subset of artificial intelligence (AI) focused on developing and studying statistical algorithms that enable systems to learn patterns from data. Unlike traditional programming, where explicit instructions are coded, ML models generalize from examples (data) to perform tasks, even when encountering new, unseen information. For instance, an ML model trained on thousands of cat and dog images can later classify a new photo as a cat or dog without being hard-coded for that specific task. Key aspects include:  

1. **Learning from Data**: Models improve their performance by analyzing training data.  
2. **Generalizati

{'input': 'What is Machine Learning?',
 'output': 'Machine Learning (ML) is a subset of artificial intelligence (AI) focused on developing and studying statistical algorithms that enable systems to learn patterns from data. Unlike traditional programming, where explicit instructions are coded, ML models generalize from examples (data) to perform tasks, even when encountering new, unseen information. For instance, an ML model trained on thousands of cat and dog images can later classify a new photo as a cat or dog without being hard-coded for that specific task. Key aspects include:  \n\n1. **Learning from Data**: Models improve their performance by analyzing training data.  \n2. **Generalization**: They adapt to new, previously unseen inputs by identifying underlying patterns.  \n3. **Task Automation**: ML powers applications like image recognition, language translation, recommendation systems, and more.  \n\nIt bridges the gap between human-like learning and computational efficiency, 

In [25]:
executor.invoke({
    "input": "What is the paper 1706.03762 about?"
})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on cThe paper with ID **1706.03762** is titled **"Attention Is All You Need"** and was authored by Ashish Vaswani, Noam Shazeer, Niki Parmar, and others. It is a foundational work in natural language processing (NLP) that introduces the **Transformer architecture**, which relies entirely on self-attention mechanisms to process sequences. This approach eliminates the need for recurrence or convolution, enabling parallel processing and achieving state-of-the-art results in tasks like machine translation.

### Key Points:
- **Publication Date:** The tool response lists 2023, but this appears to be incorrect. The paper was originally published in **2017** (a

{'input': 'What is the paper 1706.03762 about?',
 'output': 'The paper with ID **1706.03762** is titled **"Attention Is All You Need"** and was authored by Ashish Vaswani, Noam Shazeer, Niki Parmar, and others. It is a foundational work in natural language processing (NLP) that introduces the **Transformer architecture**, which relies entirely on self-attention mechanisms to process sequences. This approach eliminates the need for recurrence or convolution, enabling parallel processing and achieving state-of-the-art results in tasks like machine translation.\n\n### Key Points:\n- **Publication Date:** The tool response lists 2023, but this appears to be incorrect. The paper was originally published in **2017** (as indicated by the arXiv identifier `1706.03762`).\n- **Impact:** The Transformer model introduced in this paper revolutionized NLP, forming the basis for architectures like BERT, GPT, and other large language models.\n- **Summary (partial):** The paper argues that attention me